In [1]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

In [2]:
llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai")

In [3]:
@tool()
def add(a: int, b: int) -> int:
    """Adds a and b."""
    return a + b

@tool()
def multiply(a: int, b: int) -> int:
    """Multiplies a and b."""
    return a * b

In [4]:
tools = [add, multiply]
llm_with_tools = llm.bind_tools(tools)

query = "Add 10 and 20 and then multiply it with 40"
# query = "Add 10 and 20, multiply 20 and 40"
# query = "What is the capital of Spain"

messages = [
    SystemMessage("You can use tools when necessary, but otherwise answer the user using your own knowledge."),
    HumanMessage(query),
]

In [5]:

while True:
    ai_msg = llm_with_tools.invoke(messages)
    # Add AI message to messages
    messages.append(ai_msg)

    # Terminate loop if no tool is to be called
    if len(ai_msg.tool_calls) == 0:
        break

    # call tool selected by LLM
    for tool_call in ai_msg.tool_calls:
        selected_tool = eval(tool_call["name"])
        tool_result = selected_tool.invoke(tool_call)
        messages.append(tool_result)

In [6]:
for message in messages:
    message.pretty_print()

================================ System Message ================================

You can use tools when necessary, but otherwise answer the user using your own knowledge.
================================ Human Message =================================

Add 10 and 20 and then multiply it with 40
================================== Ai Message ==================================
Tool Calls:
  add (76735a63-2fa2-43bc-a027-bed49e2a47ea)
 Call ID: 76735a63-2fa2-43bc-a027-bed49e2a47ea
  Args:
    a: 10
    b: 20
================================= Tool Message =================================
Name: add

30
================================== Ai Message ==================================
Tool Calls:
  multiply (230ccf5a-6e08-4c83-9561-0d729d8fe7d8)
 Call ID: 230ccf5a-6e08-4c83-9561-0d729d8fe7d8
  Args:
    a: 30
    b: 40
================================= Tool Message =================================
Name: multiply

1200
================================== Ai Message =============================